In [0]:
%sql
-- Criando o catálogo pra organizar tudo do projeto e não misturar os dados
CREATE CATALOG IF NOT EXISTS medalhao

In [0]:
%sql 
-- Tudo o que vier depois estará dentro do catálogo 'medalhao'
USE CATALOG medalhao;

-- Criando a pasta 'bronze' se ela ainda não existir, pra eu ter onde jogar os dados brutos
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
#Listando os arquivos que carreguei no Volume pra conferir se os CSVs chegaram mesmo
dbutils.fs.ls("/Volumes/medalhao/bronze/olist_raw/")

In [0]:
%sql
-- Por garantia de funcionamento testei se o código funcionou

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# Trazendo as funções do Spark pra tratar as colunas e as datas depois
from pyspark.sql import functions as F

# Guardando as pastas em variáveis, caso mude um nome no futuro, não precisar caçar erro no código todo
catalogo = "medalhao"
schema = "bronze"
volume = "olist_raw"

# Montando o caminho de onde os arquivos CSV estão salvos
volume_path = f"/Volumes/{catalogo}/{schema}/{volume}"

In [0]:
# Essa função foi feita pra automatizar a carga e não ter que ficar repetindo código pros 9 arquivos
def ingest_csv(nome_arquivo, nome_tabela):
    try:
        print(f"Iniciando a ingestão do arquivo{nome_arquivo}")

#Lendo o CSV: avisando que tem cabeçalho e pedindo pro Spark tratar os tipos de dados sozinho
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(f"{volume_path}/{nome_arquivo}")

# Criando a coluna de tempo pra saber exatamente quando esse dado entrou no sistema
        df_final = df.withColumn("timestamp_ingestion", F.current_timestamp())

# Salvando na Bronze, se a tabela já existir, ele sobrescreve com os dados novos
        df_final.write.mode("overwrite").saveAsTable(f"{schema}.{nome_tabela}")

        print(f"Tabela {schema}.{nome_tabela} criada com sucesso!")

#ele me avisa aqui sem travar o resto do código, caso de algum erro no codigo
    except Exception as e:
        print(f"Erro ao processar {nome_arquivo}: {str(e)}")







In [0]:
# Chamei a função passando o arquivo dos clientes e o nome da tabela para rodar
ingest_csv("olist_customers_dataset.csv","tb_customers")

In [0]:
# Chamei a função passando o arquivo da geolocalização e o nome da tabela para rodar
ingest_csv("olist_geolocation_dataset.csv", "tb_geolocalizacao")

In [0]:
# Chamei a função passando o resto dos arquivos "csvs" e o nome da tabela para rodar
ingest_csv("olist_order_items_dataset.csv", "tb_order_items")
ingest_csv("olist_order_payments_dataset.csv","tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "tb_orders")
ingest_csv("olist_products_dataset.csv", "tb_products")
ingest_csv("olist_sellers_dataset.csv", "tb_sellers")
ingest_csv("product_category_name_translation.csv", "tb_product_category_name_translation") 

In [0]:
# Pegando a tabela de clientes pra conferir os dados
df_check = spark.table("bronze.tb_customers")

# Mostrando só as 5 primeiras linhas pra ver se as cidades e o horário de carga tão certas
display(df_check.select("customer_id", "customer_city", "timestamp_ingestion").limit(5))

In [0]:
# Criando as caixinhas de texto no topo do notebook pra alterar as datas da consulta sem mexer no código
dbutils.widgets.text("data_inicio", "01-01-2023") 
dbutils.widgets.text("data_fim", "12-31-2023")

In [0]:
# Trazendo as bibliotecas pra bater na API e transformar os dados de Pandas pra Spark
import requests
import requests
import pandas as pd
from pyspark.sql import functions as F

# Pegando as datas que preenchi nas caixinhas do topo do notebook
data_ini = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Montando a URL do Banco Central com as datas dinâmicas pra pegar o dólar PTAX
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_ini}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

try:
    # Fazendo a requisição pra API e pegando só o 'value' do JSON que interessa
    response = requests.get(url)
    data = response.json()['value'] 
    
    # Criando um DataFrame temporário em Pandas e depois convertendo pro Spark conseguir ler
    df_pd = pd.DataFrame(data)
    df_spark = spark.createDataFrame(df_pd)
    
    # Adicionando o tempo da ingestão pra manter o padrão da Bronze
    df_bronze_cotacao = df_spark.withColumn("timestamp_ingestion", F.current_timestamp())
    
    # Salvando a cotação na tabela tb_cotacao_dolar dentro do schema
    df_bronze_cotacao.write.mode("overwrite").saveAsTable(f"{schema}.tb_cotacao_dolar")
    
    print(f" Tabela {schema}.tb_cotacao_dolar criada com {df_bronze_cotacao.count()} registros.")

except Exception as e:
    print(f"Erro na ingestão da API: {e}")

In [0]:
%sql
-- Para conferir se os 9 CSVs e a tabela do dólar tão todos criados na Bronze
SHOW TABLES IN bronze;

In [0]:
#Para conferir se os dados da API chegaram na tabela final
display(spark.table("bronze.tb_cotacao_dolar").limit(5))

In [0]:
#Conferir na estrutura e ver se os tipos de dados (string, double, etc) da API ficaram certinhos
spark.table("bronze.tb_cotacao_dolar").printSchema()